Packages importeren en zorgen dat output niet wordt onderbroken door puntjes

In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import altair as alt
import pickle
import gdown

Bestand items ophalen van google drive

In [ ]:
shared_url = f'https://drive.google.com/file/d/10Q0ZCqkqt91xQU4VRvFHdatxNkUi8A2E/view?usp=sharing'

file_id = shared_url.split('/d/')[1].split('/')[0]
download_url = f'https://drive.google.com/uc?id={file_id}'

output = 'items.parquet'
gdown.download(download_url, output, quiet=False)

items = pd.read_parquet(output)
print(items.head())


Downloading...
From: https://drive.google.com/uc?id=10Q0ZCqkqt91xQU4VRvFHdatxNkUi8A2E
To: c:\Users\m.muller\Documents\Python\Projects\Supermarket CFGS\Code\items.parquet
100%|██████████| 25.9k/25.9k [00:00<00:00, 11.8MB/s]


   item_nbr        family  class  perishable
0     96995     GROCERY I   1093           0
1     99197     GROCERY I   1067           0
2    103501      CLEANING   3008           0
3    103520     GROCERY I   1028           0
4    103665  BREAD/BAKERY   2712           1


Bestanden history per year ophalen en samenvoegen

In [2]:
# tuple: (google drive link, month)
shared_urls = [('https://drive.google.com/file/d/1Bjq3Z1CxachXIk2xos_i3VDThIH_w0bk/view?usp=drive_link', '2013-01'),
               ('https://drive.google.com/file/d/1_3VqSZ8elL04phadTBTNeAHoRTyqIdDH/view?usp=drive_link', '2013-02'),
               ('https://drive.google.com/file/d/1mV4xYuJxvXCn4S2YJq5ZFYGjAVoqwe8D/view?usp=drive_link', '2013-03'),
               ('https://drive.google.com/file/d/1tG0ONMOfvdJ_mMJxjnA1E_pKNF1oxhbV/view?usp=drive_link', '2013-04'),
               ('https://drive.google.com/file/d/1xXE3i_wVDaL8GslgFFFbwEhrvz1FlbFC/view?usp=drive_link', '2013-05'),
               ('https://drive.google.com/file/d/1VX63bRKjsVuQA7G8kKJRZ8r1N3t4DUk9/view?usp=drive_link', '2013-06'),
               ('https://drive.google.com/file/d/1SHRDDdhjPXeiAeTOqn97FbNHDRujA5va/view?usp=drive_link', '2013-07'),
               ('https://drive.google.com/file/d/1vet_rBta3ki0lGSv4bd8rzw-3UclTvHL/view?usp=drive_link', '2013-08'),
               ('https://drive.google.com/file/d/1sPAXklBLeBc7F0yPTO7lC1IYXXWQam7m/view?usp=drive_link', '2013-09'),
               ('https://drive.google.com/file/d/1cpEHEO1ufb_gm_8ropoMJsyLaj0E-FLi/view?usp=drive_link', '2013-10'),
               ('https://drive.google.com/file/d/1ahTiXlgGhqbBT7h9N31ShuhHh4XFIHPD/view?usp=drive_link', '2013-11'),
               ('https://drive.google.com/file/d/16zcBKzjv2bxOjr-LKOt3CgxRNb1J6iaj/view?usp=drive_link', '2013-12'),
               ('https://drive.google.com/file/d/1g9dFpLrxQIhL7RviPl_748N4aHizvH8r/view?usp=drive_link', '2014-01'),
               ('https://drive.google.com/file/d/1zVnUjBpjyi2dictNEv5pgi6UwPEkgjLg/view?usp=drive_link', '2014-02'),
               ('https://drive.google.com/file/d/196iDpOtX-u-PRGcaVZxv1OEWxrIzNl00/view?usp=drive_link', '2014-03'),
               ('https://drive.google.com/file/d/1LPd9-oXITgIyYWVgRk9tCdzc3kLyH5-q/view?usp=drive_link', '2014-04'),
               ('https://drive.google.com/file/d/10smcu-uIHABHsNWJ6nWiB51zlxnlp6EO/view?usp=drive_link', '2014-05'),
               ('https://drive.google.com/file/d/13L8tiHMlppRWbNbykdj8C13kwagFzFoY/view?usp=drive_link', '2014-06'),
               ('https://drive.google.com/file/d/1d6PlDS4hTpd6DCI4KpTb7GJ4yfwe1jad/view?usp=drive_link', '2014-07'),
               ('https://drive.google.com/file/d/1V2vCtr3K8MqYdV4LTcuwCuecZ0iN0K-I/view?usp=drive_link', '2014-08'),
               ('https://drive.google.com/file/d/1Ok_kWbUYhGApCvisDDifvl3UyAP5IQJI/view?usp=drive_link', '2014-09'),
               ('https://drive.google.com/file/d/12wXpec83qgeLuU5SJFPfdleeNGdNJtfP/view?usp=drive_link', '2014-10'),
               ('https://drive.google.com/file/d/1PeNOprlIBJKOrfeO2CnW5UwBOy4FVkTd/view?usp=drive_link', '2014-11'),
               ('https://drive.google.com/file/d/1rxgT1e5juyJ59Ljhlu9vlyzDy7Vx69Zi/view?usp=drive_link', '2014-12'),
               ('https://drive.google.com/file/d/1otP3Ba8pgzy7lwmszGQxAKSaf7XdtLqz/view?usp=drive_link', '2015-01'),
               ('https://drive.google.com/file/d/1LS-SiXXce2h0T_HGX-udwH6eal6qlOwR/view?usp=drive_link', '2015-02'),
               ('https://drive.google.com/file/d/1eo0giN7KFPZufr2FuGqOEE_twhx-OgHs/view?usp=drive_link', '2015-03'),
               ('https://drive.google.com/file/d/1KvxMXQGXkmxMHwt3JHT2ymDbzsT_9Id6/view?usp=drive_link', '2015-04'),
               ('https://drive.google.com/file/d/1dZ87mJT5Ew-KXEpn8LeQM_ROe9HeKf3N/view?usp=drive_link', '2015-05')         
               ]


sales_history = []

for link, month in shared_urls:
    file_id = link.split('/d/')[1].split('/')[0]
    download_url = f'https://drive.google.com/uc?id={file_id}'

    output = 'sales_temp.parquet'
    gdown.download(download_url, output, quiet=False)

    sales = pd.read_parquet(output)
    sales['month'] = month
    sales_history.append(sales)

    combined_sales1 = pd.concat(sales_history, ignore_index=True)

print(combined_sales1.shape)
print(combined_sales1.head())
print(combined_sales1.tail())

combined_sales1.to_parquet("combined_history_per_year_part1.parquet", index=False)


Downloading...
From: https://drive.google.com/uc?id=1Bjq3Z1CxachXIk2xos_i3VDThIH_w0bk
To: c:\Users\m.muller\Documents\Python\Projects\Supermarket CFGS\Code\sales_temp.parquet
100%|██████████| 4.44M/4.44M [00:00<00:00, 10.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1_3VqSZ8elL04phadTBTNeAHoRTyqIdDH
To: c:\Users\m.muller\Documents\Python\Projects\Supermarket CFGS\Code\sales_temp.parquet
100%|██████████| 4.20M/4.20M [00:00<00:00, 49.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1mV4xYuJxvXCn4S2YJq5ZFYGjAVoqwe8D
To: c:\Users\m.muller\Documents\Python\Projects\Supermarket CFGS\Code\sales_temp.parquet
100%|██████████| 4.74M/4.74M [00:00<00:00, 12.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1tG0ONMOfvdJ_mMJxjnA1E_pKNF1oxhbV
To: c:\Users\m.muller\Documents\Python\Projects\Supermarket CFGS\Code\sales_temp.parquet
100%|██████████| 4.62M/4.62M [00:00<00:00, 54.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1xXE3i_wVDaL8GslgFFFbwEhrvz1FlbFC
To

(47898037, 7)
   id  store_nbr  item_nbr  unit_sales  onpromotion  day    month
0   0         25    103665         7.0         <NA>    1  2013-01
1   1         25    105574         1.0         <NA>    1  2013-01
2   2         25    105575         2.0         <NA>    1  2013-01
3   3         25    108079         1.0         <NA>    1  2013-01
4   4         25    108701         1.0         <NA>    1  2013-01
                id  store_nbr  item_nbr  unit_sales  onpromotion  day    month
47898032  47898032         54   1464238        18.0        False   31  2015-05
47898033  47898033         54   1464239         6.0        False   31  2015-05
47898034  47898034         54   1464241        18.0        False   31  2015-05
47898035  47898035         54   1464309         6.0        False   31  2015-05
47898036  47898036         54   1466047         1.0        False   31  2015-05


Bestanden history per year2 ophalen en samenvoegen (wacht 20 minuten na history per year1 ophalen)

In [4]:
# tuple: (google drive link, month)
shared_urls = [('https://drive.google.com/file/d/1zERKmry3o1S7HFaZ795bRwQWCrc-b0hx/view?usp=drive_link', '2015-06'),
               ('https://drive.google.com/file/d/1ZTTfI13r1LUuiVVfH_AnYKH7xbW09ejQ/view?usp=drive_link', '2015-07'),
               ('https://drive.google.com/file/d/1nPQnMfuHhLfDzsAZl28CVGCQFVKMg0Ix/view?usp=drive_link', '2015-08'),
               ('https://drive.google.com/file/d/1YgGL8cNs9peSkVu7ZWXNW9Tv7GDGcVi2/view?usp=drive_link', '2015-09'),
               ('https://drive.google.com/file/d/16bClueKKal-45H-_x-xQrOnYOpkbf7AO/view?usp=drive_link', '2015-10'),
               ('https://drive.google.com/file/d/1sx9J9-i5uk2jfCxOr9PJTS8rtCnSczcX/view?usp=drive_link', '2015-11'),
               ('https://drive.google.com/file/d/17EPxkYdEQEEB9Gg72GUi_azoExQPJwOq/view?usp=drive_link', '2015-12'),
               ('https://drive.google.com/file/d/1ag1wnFGWv5lIwknY8JLX0CKb9hXiQpcJ/view?usp=drive_link', '2016-01'),
               ('https://drive.google.com/file/d/1ViTRl0fo4ebYwAbJUEqk3TDCqiFneaoW/view?usp=drive_link', '2016-02'),
               ('https://drive.google.com/file/d/11ex2zjM9zV57Wi_L6t_QZ9gUT4S8SCCU/view?usp=drive_link', '2016-03'),
               ('https://drive.google.com/file/d/16GuWdkWXj0So1ETlEEVJ5Ao0K4Z0WmVA/view?usp=drive_link', '2016-04'),
               ('https://drive.google.com/file/d/1xYPfjUVxGxyz4-rQXT8_OC9Iy3pz6VdI/view?usp=drive_link', '2016-05'),
               ('https://drive.google.com/file/d/1SvyjeOOCPLPifM8ubz6N11yRR57lxORx/view?usp=drive_link', '2016-06'),
               ('https://drive.google.com/file/d/114akns6a4UkH_DaaqEmWWDqGaNygBYaL/view?usp=drive_link', '2016-07'),
               ('https://drive.google.com/file/d/1cNU5GibFTIaaV4V7lm-3do8OqarmzRjw/view?usp=drive_link', '2016-08'),
               ('https://drive.google.com/file/d/1vBP-C3GUjW0WH1tRGCad24dh5-3hID2O/view?usp=drive_link', '2016-09'),
               ('https://drive.google.com/file/d/1HmX1WbXwsnkO1Q2kFp0gdMB3-SJqbqr4/view?usp=drive_link', '2016-10'),
               ('https://drive.google.com/file/d/1JkhMtkHQhKBJaA2twhuDuKXd2Nt5-8kJ/view?usp=drive_link', '2016-11'),
               ('https://drive.google.com/file/d/1KnrX6E8zoNah16hrw2xfOj2b_uGWYmHg/view?usp=drive_link', '2016-12'),
               ('https://drive.google.com/file/d/1vVas6aIHBXfSy8f9kPADs0LsTYBrjsq5/view?usp=drive_link', '2017-01'),
               ('https://drive.google.com/file/d/1lYo-qaZzY1wRVY0KEJZwR4mBHPl6eRAy/view?usp=drive_link', '2017-02'),
               ('https://drive.google.com/file/d/1mxp0_ONXtf45SBYDcycw9kdqnzy2OU_b/view?usp=drive_link', '2017-03'),
               ('https://drive.google.com/file/d/1g2al11GA_vAuGqSkWCHo2kInfplibHmz/view?usp=drive_link', '2017-04'),
               ('https://drive.google.com/file/d/1_OvmNX9vacmQDk7Oz9KaOyErB6jKSQ-h/view?usp=drive_link', '2017-05'),
               ('https://drive.google.com/file/d/1LIy7BxSRg6R2wRiSeWttnt49EfFgT6JQ/view?usp=drive_link', '2017-06'),
               ('https://drive.google.com/file/d/1Qsb-oP2mCeeEGdm4efHzCp_R6fKUNUtn/view?usp=drive_link', '2017-07'),
               ('https://drive.google.com/file/d/1pp3ISLxcNlpR4Y1cg0mS3CmfCwQmL8k8/view?usp=drive_link', '2017-08')          
               ]


sales_history = []

for link, month in shared_urls:
    file_id = link.split('/d/')[1].split('/')[0]
    download_url = f'https://drive.google.com/uc?id={file_id}'

    output = 'sales_temp2.parquet'
    gdown.download(download_url, output, quiet=False)

    sales = pd.read_parquet(output)
    sales['month'] = month
    sales_history.append(sales)

    combined_sales2 = pd.concat(sales_history, ignore_index=True)

combined_sales = pd.concat([combined_sales1, combined_sales2], ignore_index=True)

print(combined_sales.shape)
print(combined_sales.head())
print(combined_sales.tail())

combined_sales.to_parquet("combined_history_per_year.parquet", index=False)


Downloading...
From: https://drive.google.com/uc?id=1zERKmry3o1S7HFaZ795bRwQWCrc-b0hx
To: c:\Users\m.muller\Documents\Python\Projects\Supermarket CFGS\Code\sales_temp2.parquet
100%|██████████| 7.85M/7.85M [00:00<00:00, 69.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ZTTfI13r1LUuiVVfH_AnYKH7xbW09ejQ
To: c:\Users\m.muller\Documents\Python\Projects\Supermarket CFGS\Code\sales_temp2.parquet
100%|██████████| 8.28M/8.28M [00:00<00:00, 68.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1nPQnMfuHhLfDzsAZl28CVGCQFVKMg0Ix
To: c:\Users\m.muller\Documents\Python\Projects\Supermarket CFGS\Code\sales_temp2.parquet
100%|██████████| 8.63M/8.63M [00:00<00:00, 57.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1YgGL8cNs9peSkVu7ZWXNW9Tv7GDGcVi2
To: c:\Users\m.muller\Documents\Python\Projects\Supermarket CFGS\Code\sales_temp2.parquet
100%|██████████| 8.64M/8.64M [00:00<00:00, 65.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=16bClueKKal-45H-_x-xQrOnYOpkbf7A

(125497040, 7)
   id  store_nbr  item_nbr  unit_sales  onpromotion  day    month
0   0         25    103665         7.0         <NA>    1  2013-01
1   1         25    105574         1.0         <NA>    1  2013-01
2   2         25    105575         2.0         <NA>    1  2013-01
3   3         25    108079         1.0         <NA>    1  2013-01
4   4         25    108701         1.0         <NA>    1  2013-01
                  id  store_nbr  item_nbr  unit_sales  onpromotion  day  \
125497035  125497035         54   2089339         4.0        False   15   
125497036  125497036         54   2106464         1.0         True   15   
125497037  125497037         54   2110456       192.0        False   15   
125497038  125497038         54   2113914       198.0         True   15   
125497039  125497039         54   2116416         2.0        False   15   

             month  
125497035  2017-08  
125497036  2017-08  
125497037  2017-08  
125497038  2017-08  
125497039  2017-08  


Opslaan pickle file

In [5]:
# Create dictionary 'dc_exercise_1_2_3' with objects that will be used in the next exercises.
dc_sales_history = {
'combined_sales': combined_sales,
}

# Save dc_exercise_1_2_3 as 'dc-ames-housing-pieter-exercise-1-2-3.pkl'
with open('../data/supermarket_combined_sales.pkl', 'wb') as pickle_file:
    pickle.dump(dc_sales_history, pickle_file)